# 03 — vLLM Fork Setup

This notebook prepares the environment to run notebook 04 against a
modified (forked) version of vLLM with Python-only changes.

Run this once before executing `04_vllm_niah.ipynb` with the fork.

## Configuration

In [ ]:
FORK_URL = "https://github.com/fax4ever/vllm.git"
FORK_BRANCH = "fax-0.18"
FORK_DIR = "/opt/app-root/src/vllm-fork"

## Step 1 — Clone the fork to persistent storage

In [ ]:
import os
import shutil

if os.path.exists(FORK_DIR):
    shutil.rmtree(FORK_DIR)
    print(f"Removed previous clone at {FORK_DIR}")

!git clone {FORK_URL} {FORK_DIR}
!cd {FORK_DIR} && git checkout {FORK_BRANCH}
print(f"Cloned fork to {FORK_DIR} on branch {FORK_BRANCH}")

## Step 2 — Copy compiled extensions into the fork

The system vLLM contains pre-compiled C++/CUDA `.so` files that the
Python code calls into. Since the fork only modifies Python files,
we copy these compiled extensions into the fork's directory so they
remain accessible.

In [ ]:
import glob
import shutil

SYSTEM_VLLM = "/opt/app-root/lib64/python3.12/site-packages/vllm"
FORK_VLLM = os.path.join(FORK_DIR, "vllm")

so_files = glob.glob(os.path.join(SYSTEM_VLLM, "**", "*.so"), recursive=True)

copied = 0
for src in so_files:
    rel_path = os.path.relpath(src, SYSTEM_VLLM)
    dst = os.path.join(FORK_VLLM, rel_path)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    copied += 1

print(f"Copied {copied} compiled extensions from system vLLM into the fork")

## Step 3 — Verify the fork is importable

This confirms that Python can load vLLM from the fork directory.
The same `sys.path.insert` line must be added at the top of
`04_vllm_niah.ipynb` (before any `import vllm`) for the fork to
be used during the benchmark.

In [ ]:
import sys
sys.path.insert(0, FORK_DIR)

import vllm
print(f"vLLM version: {vllm.__version__}")
assert FORK_DIR in vllm.__file__, "ERROR: vLLM is NOT loading from the fork!"
print("OK — fork is active")